In [1]:
#installing dependencies

%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# creating an API client

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5"

In [ ]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system = None):
    
    params = {
        "model": model,
        "max_tokens": 1024,
        "messages": messages,
    }

    if system:
        params["system"] = system

    resp = client.messages.create(**params)

    # Normalize possible response shapes and extract text pieces
    content = None
    if hasattr(resp, 'content'):
        content = resp.content
    elif isinstance(resp, dict):
        content = resp.get('content')

    text_parts = []
    if isinstance(content, list):
        for item in content:
            # item may be a plain string
            if isinstance(item, str):
                text_parts.append(item)
                continue

            # try attribute access
            if hasattr(item, 'text'):
                try:
                    text_parts.append(getattr(item, 'text'))
                    continue
                except Exception:
                    pass

            # try dict-style access
            if isinstance(item, dict) and 'text' in item:
                text_parts.append(item['text'])
                continue

            # last resort: stringify the item
            text_parts.append(str(item))

    else:
        # Fallbacks if `content` isn't a list
        if hasattr(resp, 'text'):
            return getattr(resp, 'text')
        if isinstance(resp, dict) and 'text' in resp:
            return resp['text']
        # As a final fallback, return the string representation
        return str(resp)

    return ''.join(text_parts)

In [5]:
messages = []

system = """You are a software engineer. Your task is to write the 
code as concisely and provide an optimal solution that performs
the task given to you. 
"""
add_user_message(messages, "Write a python function that checks a string for duplicate chracters.",)
answer = chat(messages, system)

answer

'ThinkingBlock(signature=\'EroCCokBCBAYAipAaAiT7oDcsQ9WqxP0xzZCmzRFb1nf1+8kRXVIpHB9hvgzng+gnw0lKNDhK4ApWgjhkyrnFgPyStdWpd/gdjwbbzIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiRjYjdiNDY0Ni1hNmJmLTRjMGUtODc5Zi00Y2JlNjczMmY4YjcSDAvlHp36tsPHADNnFBoMLjIE9aOKFDGi+45gIjAs9UPAOaD6XbPncrDLFXmISTHlBKq8/fRchxiSC1Fr0Al4tmcnhVBKGt1MZO8J/2IqXgUcOR4E1qkZFwNCuN14ssjgAnZWmRqMXc9gh8kpQQWNe0+yhqNSfh9q/3HZyWs9TjyWrsymYkpUXbS5U4OP82+gBTKZTPJb7ybqUNfUeKnkh/uWEZiD7nXSgSEQqRoYAQ==\', thinking=\'\', type=\'thinking\')# Solution 1: Using set comparison (most concise)\ndef has_duplicates(s):\n    return len(s) != len(set(s))\n\n# Solution 2: Return the actual duplicate characters\ndef find_duplicates(s):\n    seen = set()\n    return {c for c in s if c in seen or seen.add(c)}\n\n# Example usage\nif __name__ == "__main__":\n    text = "programming"\n    print(has_duplicates(text))      # True\n    print(find_duplicates(text))     # {\'r\', \'g\', \'m\'}\n\n**Explanation:**\n- `has_duplicates`: Converts the string to a 